[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]
(https://colab.research.google.com/github/JoneSu1/DeepCompare_batch_mark/blob/main/notebooks/run_colab.ipynb)

# MPRA benchmark on Colab — DeepCompARE vs Enformer vs AlphaGenome

1. **Runtime → Change runtime type → T4 GPU** (Enformer needs it; DeepCompARE runs on CPU).
2. Edit `REPO_URL` in the first cell to your GitHub repo (HTTPS, public or with token).
3. Run all cells. AlphaGenome asks for an API key via the form (free, non-commercial:
   https://deepmind.google.com/science/alphagenome).

In [1]:
REPO_URL = "https://github.com/JoneSu1/DeepCompare_batch_mark"
!git clone -q {REPO_URL} mpra-benchmark
%cd mpra-benchmark

/content/mpra-benchmark


In [2]:
!pip install -q -r requirements.txt
!pip install -q --no-deps "git+https://github.com/anderssonlab/deepISA"
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('' + '!' * 70)
    print('NO GPU DETECTED. Enformer will refuse to run on CPU (~34 h).')
    print('Fix now: 代码执行程序/Runtime -> 更改运行时类型/Change runtime type -> T4 GPU -> 保存/Save')
    print('then Run all again.')
    print('!' * 70 + '')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.9/213.9 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 130.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Mount Google Drive: results are saved to PhD_sub_project/Project10_xuening/output
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Step 1 — sequences (downloads the 6 needed hg38 chromosomes from UCSC, ~5 min)

Always run this for the **full** benchmark: the repo ships MVP-only artifacts
(HBB+HBG1, 606 variants) and this step overwrites them with the full 3,877-variant set.

In [4]:
import os
os.makedirs('data/reference', exist_ok=True)
for c in [1, 2, 11, 19, 22, 'X']:
    path = f'data/reference/chr{c}.fa'
    if not os.path.exists(path):
        !curl -sL "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr{c}.fa.gz" | gunzip > {path}
    print(path, 'ready')
!python scripts/01_prepare_sequences.py --fasta data/reference/chr1.fa data/reference/chr2.fa data/reference/chr11.fa data/reference/chr19.fa data/reference/chr22.fa data/reference/chrX.fa

data/reference/chr1.fa ready
data/reference/chr2.fa ready
data/reference/chr11.fa ready
data/reference/chr19.fa ready
data/reference/chr22.fa ready
data/reference/chrX.fa ready
{
  "tsv": "GRCh38_F9_GP1BA_HBB_HBG1_LDLR_PKLR-24h_BCL11A_SORT1.tsv",
  "filters": {
    "alt_not_dash": true,
    "p_value_max": 0.05
  },
  "n_variants": 3877,
  "per_element": {
    "SORT1": 1155,
    "PKLR-24h": 521,
    "LDLR": 486,
    "GP1BA": 408,
    "HBG1": 362,
    "F9": 359,
    "BCL11A": 342,
    "HBB": 244
  },
  "elements_written": [
    "BCL11A",
    "F9",
    "GP1BA",
    "HBB",
    "HBG1",
    "LDLR",
    "PKLR-24h",
    "SORT1"
  ]
}


In [5]:
!python scripts/02_predict_deepcompare.py --device cpu

BCL11A     K562   dc_t1 (CAGE K562): r = +0.088  (n=342)
BCL11A     K562   dc_t3 (DNase K562): r = +0.048  (n=342)
BCL11A     K562   dc_t5 (STARR K562): r = +0.188  (n=342)
BCL11A     K562   dc_t7 (SuRE K562): r = +0.092  (n=342)
F9         HepG2  dc_t0 (CAGE HepG2): r = +0.434  (n=359)
F9         HepG2  dc_t2 (DNase HepG2): r = +0.471  (n=359)
F9         HepG2  dc_t4 (STARR HepG2): r = +0.731  (n=359)
F9         HepG2  dc_t6 (SuRE HepG2): r = +0.766  (n=359)
GP1BA      K562   dc_t1 (CAGE K562): r = +0.365  (n=408)
GP1BA      K562   dc_t3 (DNase K562): r = +0.517  (n=408)
GP1BA      K562   dc_t5 (STARR K562): r = +0.371  (n=408)
GP1BA      K562   dc_t7 (SuRE K562): r = +0.546  (n=408)
HBB        K562   dc_t1 (CAGE K562): r = +0.645  (n=244)
HBB        K562   dc_t3 (DNase K562): r = +0.541  (n=244)
HBB        K562   dc_t5 (STARR K562): r = +0.190  (n=244)
HBB        K562   dc_t7 (SuRE K562): r = +0.687  (n=244)
HBG1       K562   dc_t1 (CAGE K562): r = +0.345  (n=362)
HBG1       K562   d

In [8]:
# Enformer on T4: ~3,885 contexts (3,877 mut + 8 unique refs), expect ~15-30 min total
# T4 has 16 GB VRAM; if you hit a CUDA OOM, rerun with --batch-size 4
!python scripts/03_predict_enformer.py --device auto --batch-size 4


2026-09-30 08:54:45.752158: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
device=cuda | variants=3877 | tracks=13 | center_bin=448
  4/8 seqs (08:54:56)
  4/3877 seqs (08:55:06)
  44/3877 seqs (08:55:53)
  84/3877 seqs (08:56:48)
  124/3877 seqs (08:57:42)
  164/3877 seqs (08:58:37)
  204/3877 seqs (08:59:31)
  244/3877 seqs (09:00:26)
  284/3877 seqs (09:01:20)
  324/3877 seqs (09:02:14)
  364/3877 seqs (09:03:09)
  404/3877 seqs (09:04:03)
  444/3877 seqs (09:04:58)
  484/3877 seqs (09:05:52)
  524/3877 seqs (09:06:47)
  564/3877 seqs (09:07:41)
  604/3877 seqs (09:08:36)
  644/3877 seqs (09:09:30)
  684/3877 seqs (09:10:25)
  724/3877 seqs (09:11:19)
  764/3877 seqs (09:12:14)
  804/3877 seqs (09:13:08)
  844/3877 seqs (09:14:03)
  884/3877 

## Step 4 — AlphaGenome (API; ~3,885 calls, resumable via SQLite cache)

In [9]:
# @title Fill in your AlphaGenome API key
API_KEY = ""  # @param {type:"string"}
assert API_KEY, 'paste your key from https://deepmind.google.com/science/alphagenome'
with open('configs/alphagenome.yaml', 'w') as f:
    f.write(open('configs/alphagenome.yaml.example').read().replace('${ALPHAGENOME_API_KEY}', API_KEY))
!python scripts/04_predict_alphagenome.py

2026-09-30 10:58:37.208 | INFO     | deepISA.model.alpha_genome_adapter:__init__:360 - [AlphaGenome] persistent cache: /content/mpra-benchmark/configs/alphagenome.cache.sqlite (0 existing predictions will be reused)
refs done: 8 seqs in 2s
2026-09-30 11:07:26.727 | INFO     | deepISA.model.alpha_genome_adapter:_maybe_log_stats:622 - [AlphaGenome] progress: api_calls=1000 cache_hits=0 retries=0 cache_entries=998
2026-09-30 11:16:44.059 | INFO     | deepISA.model.alpha_genome_adapter:_maybe_log_stats:622 - [AlphaGenome] progress: api_calls=2000 cache_hits=0 retries=0 cache_entries=1998
2026-09-30 11:23:46.778 | INFO     | deepISA.model.alpha_genome_adapter:_maybe_log_stats:622 - [AlphaGenome] progress: api_calls=3000 cache_hits=0 retries=0 cache_entries=2998
muts done: 3877 seqs in 1952s | stats: {'api_calls': 3886, 'api_successes': 3886, 'cache_hits': 0, 'cache_misses': 3885, 'retries': 0, 'quota_waits': 0, 'quota_wait_seconds': 0.0, 'client_rebuilds': 0, 'invalid_argument_recoveries': 

In [10]:
!python scripts/05_correlations.py

 Element  Cell      Method       Track  Correlation    N
  BCL11A  K562 DeepCompARE  CAGE HepG2     0.101758  342
  BCL11A  K562 DeepCompARE   CAGE K562     0.087586  342
  BCL11A  K562 DeepCompARE DNase HepG2     0.070937  342
  BCL11A  K562 DeepCompARE  DNase K562     0.048164  342
  BCL11A  K562 DeepCompARE STARR HepG2     0.182274  342
  BCL11A  K562 DeepCompARE  STARR K562     0.187965  342
  BCL11A  K562 DeepCompARE  SuRE HepG2     0.083460  342
  BCL11A  K562 DeepCompARE   SuRE K562     0.091959  342
      F9 HepG2 DeepCompARE  CAGE HepG2     0.434496  359
      F9 HepG2 DeepCompARE   CAGE K562     0.536740  359
      F9 HepG2 DeepCompARE DNase HepG2     0.471000  359
      F9 HepG2 DeepCompARE  DNase K562     0.212225  359
      F9 HepG2 DeepCompARE STARR HepG2     0.731181  359
      F9 HepG2 DeepCompARE  STARR K562     0.654082  359
      F9 HepG2 DeepCompARE  SuRE HepG2     0.766433  359
      F9 HepG2 DeepCompARE   SuRE K562     0.673362  359
   GP1BA  K562 DeepCompARE  CAG

In [11]:
# Save results to Google Drive with a timestamp (previous runs are kept)
import os, shutil
from datetime import datetime
import pandas as pd
corr = pd.read_csv('results/correlations.csv')
display(corr.pivot_table(index='Element', columns=['Method', 'Track'], values='Correlation').round(2))

OUT = '/content/drive/MyDrive/PhD_sub_project/Project10_xuening/output'
os.makedirs(OUT, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M')
for f in ['results/correlations.csv',
          'results/predictions/deepcompare.csv',
          'results/predictions/enformer.csv',
          'results/predictions/alphagenome.csv']:
    if os.path.exists(f):
        dst = os.path.join(OUT, f'{stamp}_{os.path.basename(f)}')
        shutil.copy(f, dst)
        print('saved ->', dst)
    else:
        print(f'skipped (not produced): {f}')


Method   AlphaGenome                                                        \
Track     ATAC HepG2 ATAC K562 CAGE HepG2 CAGE K562 DNASE HepG2 DNASE K562   
Element                                                                      
BCL11A          0.19      0.15       0.13      0.10        0.18       0.16   
F9              0.39      0.50       0.50      0.61        0.47       0.54   
GP1BA           0.55      0.64       0.60      0.58        0.62       0.54   
HBB             0.50      0.64       0.22      0.64        0.47       0.59   
HBG1           -0.11      0.52       0.18      0.55       -0.20       0.52   
LDLR            0.73      0.72       0.72      0.72        0.72       0.74   
PKLR-24h        0.35      0.68       0.55      0.70        0.52       0.69   
SORT1           0.69      0.35       0.69      0.33        0.74       0.37   

Method   DeepCompARE                                                          \
Track     CAGE HepG2 CAGE K562 DNase HepG2 DNase K562 STARR HepG2 STARR K562   
Element                                                                        
BCL11A          0.10      0.09        0.07       0.05        0.18       0.19   
F9              0.43      0.54        0.47       0.21        0.73       0.65   
GP1BA           0.34      0.37        0.35       0.52        0.30       0.37   
HBB             0.31      0.65        0.39       0.54        0.06       0.19   
HBG1            0.39      0.34        0.37       0.45       -0.02       0.08   
LDLR            0.30      0.39        0.45       0.49        0.38       0.44   
PKLR-24h        0.44      0.55        0.25       0.58        0.17       0.44   
SORT1           0.11      0.16        0.53       0.21        0.32       0.22   

Method                          Enformer                                   
Track    SuRE HepG2 SuRE K562 CAGE HepG2 CAGE K562 DNASE HepG2 DNASE K562  
Element                                                                    
BCL11A         0.08      0.09      -0.08     -0.13        0.15       0.16  
F9             0.77      0.67       0.19      0.02        0.31       0.26  
GP1BA          0.45      0.55       0.30      0.45        0.32       0.49  
HBB            0.51      0.69      -0.29      0.36        0.54       0.63  
HBG1           0.39      0.52      -0.64     -0.39       -0.39       0.16  
LDLR           0.56      0.54       0.49      0.50        0.59       0.58  
PKLR-24h       0.42      0.63      -0.12      0.58       -0.04       0.79  
SORT1          0.42      0.16       0.59      0.00        0.67       0.23

saved -> /content/drive/MyDrive/PhD_sub_project/Project10_xuening/output/20260930_1132_correlations.csv
saved -> /content/drive/MyDrive/PhD_sub_project/Project10_xuening/output/20260930_1132_deepcompare.csv
saved -> /content/drive/MyDrive/PhD_sub_project/Project10_xuening/output/20260930_1132_enformer.csv
saved -> /content/drive/MyDrive/PhD_sub_project/Project10_xuening/output/20260930_1132_alphagenome.csv
